In [1]:
# 6-21-2026

In [41]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import glob
import os

In [12]:
def prep_features(X_raw, scaler):
    # scales with the domain's shared scaler, then drops the redundant lccs column for nn input only
    X_scaled = pd.DataFrame(scaler.transform(X_raw), columns=X_raw.columns, index=X_raw.index)
    return X_scaled.drop(columns=["lccs_class_2"]).values

In [27]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
# simple model since rf/xgb showed low signal cieling
    model.compile(optimizer="adam", loss="mse")
    return model

In [34]:
domain_id = "22"

X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
X_train_nn = prep_features(X_train, scaler)
X_test_nn = prep_features(X_test, scaler)

In [35]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_nn, y_train, test_size=0.2, random_state=5
)

In [36]:
NN_PARAMS = {
    "epochs": 100,
    "batch_size": 256,
    "patience": 10 # if loss doesnt imporve over this many epochs, stop
}

In [37]:
tf.random.set_seed(5)
model = build_model(input_dim=X_tr.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=NN_PARAMS["patience"],
    restore_best_weights=True
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=NN_PARAMS["epochs"],
    batch_size=NN_PARAMS["batch_size"],
    callbacks=[early_stop],
    verbose=0
)

In [38]:
print(f"stopped at epoch: {len(history.history['loss'])}")

stopped at epoch: 69


In [39]:
val_pred = model.predict(X_val, verbose=0).flatten()
test_pred = model.predict(X_test_nn, verbose=0).flatten()
train_pred = model.predict(X_tr, verbose=0).flatten()


val_spearman, _ = spearmanr(y_val, val_pred)
test_spearman, _ = spearmanr(y_test, test_pred)
train_spearman, _ = spearmanr(y_tr, train_pred)


print(f"val spearman: {val_spearman:.4f}, test spearman: {test_spearman:.4f}, train spearman: {train_spearman:.4f}")

val spearman: 0.3505, test spearman: 0.3462, train spearman: 0.3840


In [40]:
# initial expereimenting done, now making T_nn

In [42]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [43]:
models = {}
scalers = {}

In [ ]:
for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
    # setup train/test/scalers
    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_nn = prep_features(X_train, scaler)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_nn, y_train, test_size=0.2, random_state=5
    ) # val split

    keras.backend.clear_session()
    tf.random.set_seed(5)
    model = build_model(input_dim=X_tr.shape[1])

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=NN_PARAMS["patience"],
        restore_best_weights=True
    )

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=NN_PARAMS["epochs"],
        batch_size=NN_PARAMS["batch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done, stopped at epoch {len(model.history.history['loss'])}")
# takes ~30 min


domain 0 done, stopped at epoch 98
domain 1 done, stopped at epoch 64
domain 2 done, stopped at epoch 54
domain 4 done, stopped at epoch 54
domain 5 done, stopped at epoch 75
domain 6 done, stopped at epoch 43
domain 7 done, stopped at epoch 81
domain 8 done, stopped at epoch 77
domain 11 done, stopped at epoch 89
domain 12 done, stopped at epoch 83
domain 13 done, stopped at epoch 54
domain 16 done, stopped at epoch 82
domain 18 done, stopped at epoch 64
domain 19 done, stopped at epoch 81
domain 20 done, stopped at epoch 100
domain 21 done, stopped at epoch 40
domain 22 done, stopped at epoch 63
domain 23 done, stopped at epoch 62
domain 25 done, stopped at epoch 42
domain 26 done, stopped at epoch 67
domain 27 done, stopped at epoch 40
domain 28 done, stopped at epoch 30
domain 29 done, stopped at epoch 55
domain 30 done, stopped at epoch 98
domain 32 done, stopped at epoch 80
domain 33 done, stopped at epoch 65
domain 36 done, stopped at epoch 65
domain 37 done, stopped at epoch 1

In [45]:
test_X_raw = {}
test_y = {}

In [46]:
for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [47]:
T_spearman_nn = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [48]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_nn = prep_features(test_X_raw[j], scaler_i)
        y_true = test_y[j]

        preds = model_i.predict(X_test_nn, verbose=0).flatten()
        T_spearman_nn.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"evaluated source domain {i} against all targets")
# takes ~20 min

evaluated source domain 0 against all targets
evaluated source domain 1 against all targets
evaluated source domain 2 against all targets
evaluated source domain 4 against all targets
evaluated source domain 5 against all targets
evaluated source domain 6 against all targets
evaluated source domain 7 against all targets
evaluated source domain 8 against all targets
evaluated source domain 11 against all targets
evaluated source domain 12 against all targets
evaluated source domain 13 against all targets
evaluated source domain 16 against all targets
evaluated source domain 18 against all targets
evaluated source domain 19 against all targets
evaluated source domain 20 against all targets
evaluated source domain 21 against all targets
evaluated source domain 22 against all targets
evaluated source domain 23 against all targets
evaluated source domain 25 against all targets
evaluated source domain 26 against all targets
evaluated source domain 27 against all targets
evaluated source doma

In [49]:
T_spearman_nn

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.327229,0.257438,0.120159,0.123036,0.183881,0.053011,0.078471,0.082923,0.186895,0.051496,...,0.161021,0.128568,0.147927,0.295138,0.173672,0.024648,0.027077,0.076266,0.038750,0.031318
1,0.142303,0.304523,0.061035,0.099681,0.169610,0.057565,0.107336,0.139577,0.152684,0.072075,...,0.093961,0.009327,0.103113,0.239656,0.134745,-0.011778,-0.049085,0.106144,0.077949,0.027953
2,-0.031146,0.040359,0.265827,0.111264,-0.012327,0.085594,-0.018600,-0.040654,0.023416,0.000317,...,0.019639,0.021920,0.033017,0.035224,-0.028475,0.010882,-0.067476,0.048520,0.037771,-0.061132
4,0.215056,0.212925,0.126851,0.308778,0.193080,0.186108,0.181114,0.195278,0.187929,0.120606,...,0.177927,0.231047,0.169772,0.285454,0.217200,0.058896,0.024155,0.279515,0.138723,0.050509
5,0.077664,0.047553,0.079990,0.089583,0.367516,0.089286,0.095409,0.127508,0.180534,0.043639,...,0.084525,-0.086725,0.116447,0.248129,0.109446,-0.024253,0.134504,0.043670,0.120702,0.033253
6,0.092808,0.029335,0.126499,0.195003,0.057563,0.232624,0.088054,-0.050855,0.073795,-0.018044,...,0.042660,0.184468,0.133549,0.063904,0.182624,0.046699,0.077449,0.223194,0.069364,-0.019501
7,0.103590,0.086118,0.023097,0.152767,0.134723,0.141105,0.281547,0.043270,0.143050,0.112074,...,0.059547,0.149997,0.105315,0.141056,0.220834,0.004671,0.012608,0.154243,-0.037535,-0.044304
8,0.123538,0.103585,-0.000664,0.119731,-0.003287,0.060683,0.048076,0.345385,0.021600,-0.015370,...,0.079786,0.120152,0.026106,0.137048,0.050284,0.034234,0.028437,0.101111,0.029018,0.011751
11,0.175051,0.161579,0.079590,0.065741,0.196476,0.039552,0.122215,0.051105,0.444079,0.010464,...,0.203647,-0.080386,0.102369,0.362447,0.159043,0.017132,0.147659,0.099596,0.051022,0.017593
12,0.090072,0.145933,-0.030553,0.120405,0.163923,0.096060,0.017201,0.088108,0.122065,0.392357,...,0.082258,0.047641,0.055275,0.170815,0.083722,0.061984,0.036710,0.051041,0.085977,0.003015


In [50]:
T_spearman_nn.to_csv("transfer_matrix_spearman_nn.csv")